# Phase 3 — LSTM sequence model

Feed the last 5 gameweeks of per-GW stats, plus current fixture/market context, to predict weekly `total_points`.

**Splits:**
- Train: 2016-17 → 2023-24
- Validation: 2024-25



In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "fpl_model_dataset.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE_VAL_PREDS_PATH = RESULTS_DIR / "val_2024_25_predictions.csv"

TRAIN_SEASONS = [
    "2016-17", "2017-18", "2018-19", "2019-20",
    "2020-21", "2021-22", "2022-23", "2023-24",
]
VAL_SEASON = "2024-25"
TARGET = "total_points"
SEQ_LEN = 5
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)

Project root: C:\FPL_project
Device: cpu


In [2]:
SEQ_FEATURES = [
    "total_points", "minutes", "goals_scored", "assists",
    "expected_goals", "expected_assists", "was_home", "value",
]

STATIC_FEATURES = [
    "team_goals_scored_gw_roll5", "team_goals_conceded_gw_roll5", "team_points_gw_roll5",
    "opponent_team_points_roll5", "opponent_team_gc_roll5",
    "last_season_ppg", "last_season_minutes_share",
    "was_home", "rest_days",
    "value", "selected", "transfers_in", "transfers_out", "transfers_balance",
]

POSITION_MAP = {"GK": 1, "GKP": 1, "DEF": 2, "MID": 3, "AM": 3, "FWD": 4}


def load_modeling_table(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["position_id"] = df["position"].map(POSITION_MAP).fillna(3).astype(int)
    df["is_promoted_team"] = df["is_promoted_team"].map({True: 1.0, False: 0.0}).fillna(0.0)
    df["was_home"] = pd.to_numeric(df["was_home"], errors="coerce").fillna(0.0)
    return df


def split_sets(df: pd.DataFrame):
    train = df[df["season"].isin(TRAIN_SEASONS)].copy()
    val = df[df["season"] == VAL_SEASON].copy()
    return train, val


def scored_mask(df: pd.DataFrame) -> np.ndarray:
    return df["minutes"].fillna(0).to_numpy() > 0


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    return {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred) ** 0.5,
        "r2": r2_score(y_true, y_pred),
        "spearman": spearmanr(y_true, y_pred).statistic,
    }


def eval_both_slices(y_true: np.ndarray, y_pred: np.ndarray, played_mask: np.ndarray) -> dict:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    all_m = regression_metrics(y_true, y_pred)
    played_m = regression_metrics(y_true[played_mask], y_pred[played_mask])
    return {
        "mae_all": all_m["mae"],
        "rmse_all": all_m["rmse"],
        "r2_all": all_m["r2"],
        "spearman_all": all_m["spearman"],
        "mae_played": played_m["mae"],
        "rmse_played": played_m["rmse"],
        "r2_played": played_m["r2"],
        "spearman_played": played_m["spearman"],
    }


def metrics_row(model: str, y_true: np.ndarray, y_pred: np.ndarray, played_mask: np.ndarray) -> dict:
    row = eval_both_slices(y_true, y_pred, played_mask)
    row["model"] = model
    return row


def build_comparison_table(
    val: pd.DataFrame,
    y_true: np.ndarray,
    played_mask: np.ndarray,
    pred_specs: dict[str, str | np.ndarray],
) -> pd.DataFrame:
    rows = []
    for model, spec in pred_specs.items():
        if isinstance(spec, str):
            pred = pd.to_numeric(val[spec], errors="coerce").fillna(0.0).to_numpy(dtype=np.float64)
        else:
            pred = np.asarray(spec, dtype=np.float64)
        if len(pred) != len(y_true):
            raise ValueError(f"{model}: pred length {len(pred)} != val length {len(y_true)}")
        rows.append(metrics_row(model, y_true, pred, played_mask))
    return pd.DataFrame(rows)


def _numeric_block(df: pd.DataFrame, cols: list[str]) -> np.ndarray:
    return df[cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)


def build_lstm_arrays(df: pd.DataFrame, seq_len: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Sequences use only prior GWs within the same (season, element) group.
    df = df.sort_values(["season", "element", "gw"]).reset_index(drop=True)
    n = len(df)
    seq_dim = len(SEQ_FEATURES)
    static_dim = len(STATIC_FEATURES) + 2

    X_seq = np.zeros((n, seq_len, seq_dim), dtype=np.float32)
    X_static = np.zeros((n, static_dim), dtype=np.float32)
    y = df[TARGET].to_numpy(dtype=np.float32)

    seq_block = _numeric_block(df, SEQ_FEATURES)
    static_num = _numeric_block(df, STATIC_FEATURES)
    pos = df[["position_id"]].to_numpy(dtype=np.float32)
    prom = df[["is_promoted_team"]].to_numpy(dtype=np.float32)
    X_static = np.hstack([static_num, pos, prom])

    for _, grp in df.groupby(["season", "element"], sort=False):
        idxs = grp.index.to_numpy()
        for j, row_i in enumerate(idxs):
            start = max(0, j - seq_len)
            past = seq_block[idxs[start:j]]
            if len(past) < seq_len:
                pad = np.zeros((seq_len - len(past), seq_dim), dtype=np.float32)
                past = np.vstack([pad, past]) if len(past) else pad
            X_seq[row_i] = past

    return X_seq, X_static, y


df = load_modeling_table(DATA_PATH)
train_df, val_df = split_sets(df)
val_mask = scored_mask(val_df)

X_train_seq, X_train_static, y_train = build_lstm_arrays(train_df, SEQ_LEN)
X_val_seq, X_val_static, y_val = build_lstm_arrays(val_df, SEQ_LEN)

print(f"Train {len(train_df):,} | Val {len(val_df):,}")
print(f"Seq shape: {X_train_seq.shape} | Static dim: {X_train_static.shape[1]}")
print(f"Played rows — val {val_mask.mean():.1%}")

Train 196,538 | Val 27,605
Seq shape: (196538, 5, 8) | Static dim: 16
Played rows — val 41.9%


## LSTM model

2-layer LSTM over the last 5 GWs, concatenated with scaled static features, then a small FC head. Same training recipe as the MLP baseline (Adam, early stop on all-row Spearman).

In [5]:
class PointsLSTM(nn.Module):
    def __init__(
        self,
        seq_dim: int,
        static_dim: int,
        hidden: int = 64,
        layers: int = 2,
        dropout: float = 0.25):
        super().__init__()
        self.lstm = nn.LSTM(
            seq_dim,
            hidden,
            num_layers=layers,
            batch_first=True,
            dropout=dropout if layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.Linear(hidden + static_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x_seq: torch.Tensor, x_static: torch.Tensor) -> torch.Tensor:
        _, (h_n, _) = self.lstm(x_seq)
        h = h_n[-1]
        return self.head(torch.cat([h, x_static], dim=-1)).squeeze(-1)


def scale_lstm_inputs(
    X_train_seq: np.ndarray,
    X_train_static: np.ndarray,
    X_val_seq: np.ndarray,
    X_val_static: np.ndarray,
):
    seq_scaler = StandardScaler()
    flat_train = X_train_seq.reshape(-1, X_train_seq.shape[-1])
    seq_scaler.fit(flat_train)

    def transform_seq(x):
        n, t, d = x.shape
        out = seq_scaler.transform(x.reshape(-1, d))
        return out.reshape(n, t, d).astype(np.float32)

    static_scaler = StandardScaler()
    X_train_static_s = static_scaler.fit_transform(X_train_static).astype(np.float32)
    X_val_static_s = static_scaler.transform(X_val_static).astype(np.float32)
    return transform_seq(X_train_seq), X_train_static_s, transform_seq(X_val_seq), X_val_static_s, seq_scaler, static_scaler


def train_lstm(
    X_train_seq: np.ndarray,
    X_train_static: np.ndarray,
    y_train: np.ndarray,
    X_val_seq: np.ndarray,
    X_val_static: np.ndarray,
    y_val: np.ndarray,
    epochs: int = 80,
    batch_size: int = 4096,
    lr: float = 1e-3,
    patience: int = 6,
    dropout: float = 0.25,
):
    seq_dim = X_train_seq.shape[-1]
    static_dim = X_train_static.shape[1]
    model = PointsLSTM(seq_dim, static_dim, dropout=dropout).to(DEVICE)

    train_ds = TensorDataset(
        torch.tensor(X_train_seq, dtype=torch.float32),
        torch.tensor(X_train_static, dtype=torch.float32),
        torch.tensor(y_train, dtype=torch.float32),
    )
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    X_val_seq_t = torch.tensor(X_val_seq, dtype=torch.float32, device=DEVICE)
    X_val_static_t = torch.tensor(X_val_static, dtype=torch.float32, device=DEVICE)
    y_val_np = y_val

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
    loss_fn = nn.MSELoss()

    best_state = None
    best_val_rho = -1.0
    stale = 0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for xb_seq, xb_static, yb in loader:
            xb_seq = xb_seq.to(DEVICE)
            xb_static = xb_static.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb_seq, xb_static)
            loss = loss_fn(pred, yb)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_seq_t, X_val_static_t)
            val_pred_np = val_pred.cpu().numpy()
            val_rho = spearmanr(y_val_np, val_pred_np).statistic
            val_mae = mean_absolute_error(y_val_np, val_pred_np)

        scheduler.step(val_rho)
        history.append({
            "epoch": epoch,
            "train_mse": float(np.mean(train_losses)),
            "val_spearman_all": val_rho,
            "val_mae_all": val_mae,
        })

        if val_rho > best_val_rho:
            best_val_rho = val_rho
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_seq_t, X_val_static_t).cpu().numpy()

    metrics = eval_both_slices(y_val_np, val_pred, val_mask)
    metrics["epochs_ran"] = len(history)
    return model, val_pred, metrics, pd.DataFrame(history)


X_train_seq_s, X_train_static_s, X_val_seq_s, X_val_static_s, seq_scaler, static_scaler = scale_lstm_inputs(
    X_train_seq, X_train_static, X_val_seq, X_val_static
)

lstm_model, val_pred_lstm, lstm_metrics, train_history = train_lstm(
    X_train_seq_s,
    X_train_static_s,
    y_train,
    X_val_seq_s,
    X_val_static_s,
    y_val,
)

lstm_metrics["model"] = "mlp_lstm"
print(
    "Trained | all: MAE={mae_all:.4f} rho={spearman_all:.4f} | "
    "played: MAE={mae_played:.4f} rho={spearman_played:.4f} | epochs={epochs_ran}".format(**lstm_metrics)
)
pd.Series(lstm_metrics)

Trained | all: MAE=1.0614 rho=0.6811 | played: MAE=1.8783 rho=0.3603 | epochs=16


mae_all            1.061418
rmse_all           2.065563
r2_all             0.269565
spearman_all       0.681122
mae_played         1.878346
rmse_played          2.8009
r2_played          0.063609
spearman_played    0.360293
epochs_ran               16
model              mlp_lstm
dtype: object

## Comparison and save

Score LSTM alongside baselines from notebook `02` on the same validation rows.

In [4]:
pred_specs = {
    "roll5_points": "total_points_roll5",
    "last_season_ppg": "last_season_ppg",
    "mlp_lstm": val_pred_lstm,
}
if BASELINE_VAL_PREDS_PATH.exists():
    _base = pd.read_csv(BASELINE_VAL_PREDS_PATH)
    pred_specs["mlp_points"] = _base["pred_mlp"].to_numpy()
    pred_specs["ridge_linear"] = _base["pred_ridge"].to_numpy()

comparison = build_comparison_table(val_df, y_val, val_mask, pred_specs)
comparison.to_csv(RESULTS_DIR / "phase3_model_comparison_with_lstm.csv", index=False)

print("All validation rows:")
display(comparison[["model", "mae_all", "spearman_all", "rmse_all", "r2_all"]].sort_values("spearman_all", ascending=False))
print("Played rows only:")
display(comparison[["model", "mae_played", "spearman_played", "rmse_played", "r2_played"]].sort_values("spearman_played", ascending=False))

val_preds = val_df[["season", "element", "gw", "player_id", "name", "minutes", TARGET]].copy()
val_preds["pred_lstm"] = val_pred_lstm
if BASELINE_VAL_PREDS_PATH.exists():
    val_preds["pred_mlp"] = _base["pred_mlp"].to_numpy()
    val_preds["pred_ridge"] = _base["pred_ridge"].to_numpy()
assert len(val_preds) == len(val_df)
val_preds.to_csv(RESULTS_DIR / "val_2024_25_predictions_with_lstm.csv", index=False)

torch.save(
    {
        "model_state": lstm_model.state_dict(),
        "seq_len": SEQ_LEN,
        "seq_features": SEQ_FEATURES,
        "static_features": STATIC_FEATURES,
        "seq_scaler_mean": seq_scaler.mean_,
        "seq_scaler_scale": seq_scaler.scale_,
        "static_scaler_mean": static_scaler.mean_,
        "static_scaler_scale": static_scaler.scale_,
    },
    RESULTS_DIR / "phase3_lstm.pt",
)
print("Saved comparison, predictions, and weights to", RESULTS_DIR)

All validation rows:


,model,mae_all,spearman_all,rmse_all,r2_all
3,mlp_points,1.056206,0.686446,2.059723,0.273690
2,mlp_lstm,1.064029,0.685033,2.060226,0.273335
0,roll5_points,1.090377,0.675553,2.177385,0.188338
4,ridge_linear,1.114626,0.644458,2.105387,0.241127
1,last_season_ppg,1.552684,0.326069,2.491475,-0.062718


Played rows only:


,model,mae_played,spearman_played,rmse_played,r2_played
3,mlp_points,1.863091,0.366992,2.799487,0.064553
2,mlp_lstm,1.872134,0.361929,2.796357,0.066644
4,ridge_linear,1.870431,0.347047,2.840238,0.037121
0,roll5_points,2.081290,0.290270,3.067166,-0.122889
1,last_season_ppg,2.188108,0.226227,3.063853,-0.120464


Saved comparison, predictions, and weights to C:\FPL_project\results
